In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
from pathlib import Path
from PNW_cmap import PNW_cmap
import matplotlib.pyplot as plt
from vip_slap2_analysis.utils.utils import save_figure
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.glutamate.summary import GlutamateSummary
from vip_slap2_analysis.utils.utils import normalize
import itertools
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib notebook

In [ ]:
savepath = r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Data_Club\April2026\figures"

In [ ]:
cl, cmap, cp = PNW_cmap.get_PNW_cmap('Sailboat', n_colors=4)
cp

In [ ]:
cp = cp[::-1]

In [ ]:
target_mice = [
    803496,
    804730,804733,810196,
    809047,803121,
    826033,838410,834788
]

registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check", "volume_imaging"],
    paradigms=["change_detection_passive"],
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"Loaded {len(assets)} session assets")

In [ ]:
seq_sums = []
seq_pars = []
seq_pos = []
for asset in assets:
    try:
        derived_dir = asset.derived_dir / 'glutamate' /'glutamate_analysis'
        seq_sum = pd.read_csv(os.path.join(derived_dir, 'sequence_summary_table.csv'))
        seq_sum['dmd1_depth'] = [asset.metadata['dmd1_depth']]*len(seq_sum)
        seq_sum['dmd2_depth'] = [asset.metadata['dmd2_depth']]*len(seq_sum)
        seq_sum['session_#'] = [asset.metadata['session_#']]*len(seq_sum)
        seq_sum['session_type'] = [asset.metadata['session_type']]*len(seq_sum)
        seq_par = pd.read_parquet(derived_dir / 'sequence_per_image_table.parquet')
        seq_pos_ = pd.read_parquet(derived_dir / 'sequence_position_table.parquet')
        seq_sums.append(seq_sum)

        seq_pars.append(seq_par)
        seq_pos.append(seq_pos_)
    except:
        print(asset.session_id)

seq_summary = pd.concat(seq_sums)
seq_per_image = pd.concat(seq_pars)
seq_position = pd.concat(seq_pos)

In [ ]:
seq_paths = [glob.glob(os.path.join(asset.derived_dir,'**','glutamate_sequence_df.npz'),recursive=True)[0] for asset in assets]

In [ ]:
# per_image_path = r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Data_Club\April2026\data\sequence_per_image_summary.csv"
# seq_per_image = pd.read_csv(per_image_path)

# seq_sum_path = r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Data_Club\April2026\data\sequence_summary.csv"
# seq_summary = pd.read_csv(seq_sum_path)

In [ ]:
# Cell 3: helpers and canonical synapse key

KEY_COLS = ["session_id", "dmd", "synapse_id"]

def parse_dmd_number(x):
    """
    Robustly parse DMD labels like:
    - 'DMD1', 'DMD2'
    - 1, 2
    """
    s = str(x).strip().upper()
    if s.endswith("1"):
        return 1
    if s.endswith("2"):
        return 2
    raise ValueError(f"Could not parse DMD value: {x}")

def lookup_dmd_depth(row):
    dmd_num = parse_dmd_number(row["dmd"])
    return row["dmd1_depth"] if dmd_num == 1 else row["dmd2_depth"]

# Add a per-row depth column to the 1-row-per-synapse summary
seq_summary["dmd_depth"] = seq_summary.apply(lookup_dmd_depth, axis=1)

# Optional canonical key string for debugging / alignment
seq_summary["synapse_key"] = (
    seq_summary["session_id"].astype(str) + " | " +
    seq_summary["dmd"].astype(str) + " | " +
    seq_summary["synapse_id"].astype(str)
)

seq_per_image["synapse_key"] = (
    seq_per_image["session_id"].astype(str) + " | " +
    seq_per_image["dmd"].astype(str) + " | " +
    seq_per_image["synapse_id"].astype(str)
)

In [ ]:
# Cell 4: sanity checks on table structure

n_unique_syn_summary = seq_summary[KEY_COLS].drop_duplicates().shape[0]
n_unique_syn_per_image = seq_per_image[KEY_COLS].drop_duplicates().shape[0]

print("Rows in sequence_summary:", len(seq_summary))
print("Unique synapses in sequence_summary:", n_unique_syn_summary)

print("Rows in sequence_per_image_summary:", len(seq_per_image))
print("Unique synapses in sequence_per_image_summary:", n_unique_syn_per_image)

per_image_counts = seq_per_image.groupby(KEY_COLS).size()
print("\nPer-image rows per synapse:")
display(per_image_counts.describe())

preferred_counts = seq_per_image.groupby(KEY_COLS)["is_preferred_ranked_image"].sum()
print("\nPreferred-image flag counts per synapse:")
display(preferred_counts.value_counts().sort_index())

assert len(seq_summary) == n_unique_syn_summary, "sequence_summary should be one row per synapse"
assert (per_image_counts == 7).all(), "Expected exactly 7 per-image rows per synapse"
assert (preferred_counts == 1).all(), "Expected exactly one preferred-ranked image per synapse"

print("\nSanity checks passed.")

In [ ]:
# Cell 5: extract the preferred / highest-ranked image row for each synapse

preferred_image_df = seq_per_image.loc[
    seq_per_image["is_preferred_ranked_image"]
].copy()

# Fallback in case you ever regenerate tables without the boolean flag
if len(preferred_image_df) != len(seq_summary):
    preferred_image_df = (
        seq_per_image
        .sort_values(
            KEY_COLS + ["image_rank_within_synapse", "ranking_score"],
            ascending=[True, True, True, True, False]
        )
        .drop_duplicates(subset=KEY_COLS, keep="first")
        .copy()
    )

print("Preferred-image rows:", len(preferred_image_df))
display(preferred_image_df.head())

In [ ]:
# Cell 6: align the 1-row-per-synapse summary with the preferred-image row

preferred_cols = [
    "stimulus_name",
    "stimulus_label",
    "image_selectivity_score",
    "sequence_slope",
    "sequence_slope_norm",
    "overall_slope",
    "overall_slope_norm",
    "early_slope",
    "late_slope",
    "image_rank_within_synapse",
    "ranking_score",
    "rank_basis",
    "is_preferred_ranked_image",
]

aligned_synapse_df = seq_summary.merge(
    preferred_image_df[KEY_COLS + preferred_cols],
    on=KEY_COLS,
    how="inner",
    validate="one_to_one",
)

print("Aligned table shape:", aligned_synapse_df.shape)
display(aligned_synapse_df.head())

In [ ]:
# Cell 7: final compact analysis table with the quantities you asked for

analysis_df = aligned_synapse_df[
    [
        "session_id",
        "subject_id",
        "dmd",
        "dmd_depth",
        "synapse_id",
        "stimulus_name",
        "stimulus_label",
        "image_selectivity_score",
        "sequence_slope",
        "sequence_slope_norm",
        "overall_slope",
        "overall_slope_norm",
        "early_slope",
        "late_slope",
        "sequence_class",
        "seq_p",
        "seq_q",
        "image_rank_within_synapse",
        "ranking_score",
        "rank_basis",
    ]
].sort_values(
    ["subject_id", "session_id", "dmd", "synapse_id"]
).reset_index(drop=True)

display(analysis_df.head())
print("Final analysis_df shape:", analysis_df.shape)

In [ ]:
# Cell 8: optional quick summaries

print("Depth counts:")
display(analysis_df["dmd_depth"].value_counts(dropna=False).sort_index())

print("\nExample summary by depth:")
display(
    analysis_df.groupby("dmd_depth")[
        [
            "image_selectivity_score",
            "sequence_slope",
            "sequence_slope_norm",
            "overall_slope",
            "overall_slope_norm",
            "early_slope",
            "late_slope",
        ]
    ].agg(["mean", "median", "std", "count"])
)

In [ ]:
# Robust version: depth-binned normalized sequence slope vs selectivity
# using median + IQR, with a minimum bin size filter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plot_df = analysis_df[["image_selectivity_score", "sequence_slope_norm", "dmd_depth"]].dropna().copy()
plot_df = plot_df[plot_df["image_selectivity_score"] > 0].copy()
plot_df["log10_selectivity"] = np.log10(plot_df["image_selectivity_score"])

min_bin_n = 8
n_bins = 5

depth_map = {25:cp[0],100:cp[1],200:cp[2],250:cp[3]}

fig, ax = plt.subplots(figsize=(7, 5))

for depth, g in plot_df.groupby("dmd_depth"):
    g = g.copy()
    g["bin"] = pd.qcut(g["log10_selectivity"], q=n_bins, labels=False, duplicates="drop")

    binned = (
        g.groupby("bin")
        .agg(
            x=("log10_selectivity", "median"),
            y_med=("sequence_slope_norm", "median"),
            y_q25=("sequence_slope_norm", lambda s: s.quantile(0.25)),
            y_q75=("sequence_slope_norm", lambda s: s.quantile(0.75)),
            n=("sequence_slope_norm", "size"),
        )
        .reset_index(drop=True)
    )

    binned = binned[binned["n"] >= min_bin_n].copy()

    ax.plot(
        binned["x"],
        binned["y_med"],
        marker="o",
        lw=2,
        label=f"{int(depth)} µm",
        color = depth_map[depth]
    )
#     ax.fill_between(
#         binned["x"],
#         binned["y_q25"],
#         binned["y_q75"],
#         alpha=0.2,
#     )

ax.axhline(0, color="k", ls="--", lw=1, alpha=0.6)
ax.set_xlabel("log10(image selectivity score)")
ax.set_ylabel("Normalized sequence slope")
ax.set_title("Depth-binned normalized sequence slope")
ax.legend(title="Depth", frameon=False)
plt.tight_layout()

In [ ]:
var = []
slope = []

depth_map = {25:cp[0],100:cp[1],200:cp[2],250:cp[3]}

depth_cols = []

for sess in seq_per_image['session_id'].unique():
    sess_df = seq_per_image[seq_per_image['session_id']==sess]
    
    for dmd in sess_df['dmd'].unique():
        dmd_df = sess_df[sess_df['dmd']==dmd]
        
        for syn in dmd_df['synapse_id'].unique():
            syn_df = dmd_df[dmd_df['synapse_id']==syn]
            selectivity_var = np.max(syn_df['image_selectivity_score'])
            var.append(selectivity_var)
            
            max_slope = syn_df.loc[syn_df['early_slope'].abs().idxmax(), 'early_slope']
            
            slope.append(max_slope)
            
            syn_sum = seq_summary[(seq_summary['session_id']==sess)&(seq_summary['synapse_id']==syn)]
            
            if 'DMD1' in syn:
                syn_depth = syn_sum['dmd1_depth'].values[0]
            else:
                syn_depth = syn_sum['dmd2_depth'].values[0]
            depth_cols.append(depth_map[syn_depth])

In [ ]:
fig,ax=plt.subplots(figsize=(4,4))

sns.despine()

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)

sc = ax.scatter(np.log10(var),slope,c=depth_cols,s=5)
ax.set_xlabel('log max. selectivity score')
ax.set_ylabel('log max. |slope|')
# ax.legend(handles = sc,labels=['25','100','200','250'],frameon=False)
fig.tight_layout()

In [ ]:
slope

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rows = []

palette = {25: cp[0], 100: cp[1], 200: cp[2], 250: cp[3]}

for sess in seq_per_image['session_id'].unique():
    sess_df = seq_per_image[seq_per_image['session_id'] == sess]

    for dmd in sess_df['dmd'].unique():
        dmd_df = sess_df[sess_df['dmd'] == dmd]

        for syn in dmd_df['synapse_id'].unique():
            syn_df = dmd_df[dmd_df['synapse_id'] == syn]

            # current summaries
            max_selectivity = syn_df['image_selectivity_score'].max()
            max_abs_early_slope = np.abs(syn_df['early_slope']).max()

            syn_sum = seq_summary[
                (seq_summary['session_id'] == sess) &
                (seq_summary['synapse_id'] == syn)
            ]

            if syn_sum.empty:
                continue

            if 'DMD1' in syn:
                syn_depth = syn_sum['dmd1_depth'].iloc[0]
            else:
                syn_depth = syn_sum['dmd2_depth'].iloc[0]

            rows.append({
                'session_id': sess,
                'dmd': dmd,
                'synapse_id': syn,
                'depth': syn_depth,
                'max_selectivity': max_selectivity,
                'max_abs_early_slope': max_abs_early_slope,
                'log10_max_selectivity': np.log10(max_selectivity) if max_selectivity > 0 else np.nan,
                'log10_max_abs_early_slope': np.log10(max_abs_early_slope) if max_abs_early_slope > 0 else np.nan,
            })

plot_df = pd.DataFrame(rows).dropna()

fig, ax = plt.subplots(figsize=(4.5, 4.5))
sns.despine(ax=ax)
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
sns.scatterplot(
    data=plot_df,
    x='log10_max_selectivity',
    y='log10_max_abs_early_slope',
    hue='depth',
    hue_order=[25, 100, 200, 250],
    palette=palette,
    s=10,
    edgecolor='none',
    alpha=0.85,
    ax=ax,
)

ax.tick_params(axis='x', which='major', top=False, labelsize=12)
ax.tick_params(axis='y', which='major', right=False, labelsize=12)

for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(2)

ax.set_xlabel('log10 max selectivity score')
ax.set_ylabel('log10 max |early slope|')

leg = ax.legend(title='Depth (µm)', frameon=False)
if leg is not None:
    leg.get_title().set_fontsize(11)
    for txt in leg.get_texts():
        txt.set_fontsize(12)
    

fig.tight_layout()

In [ ]:
rows = []

palette = {25: cp[0], 100: cp[1], 200: cp[2], 250: cp[3]}

for sess in seq_per_image['session_id'].unique():
    sess_df = seq_per_image[seq_per_image['session_id'] == sess]

    for dmd in sess_df['dmd'].unique():
        dmd_df = sess_df[sess_df['dmd'] == dmd]

        for syn in dmd_df['synapse_id'].unique():
            syn_df = dmd_df[dmd_df['synapse_id'] == syn].copy()

            syn_sum = seq_summary[
                (seq_summary['session_id'] == sess) &
                (seq_summary['synapse_id'] == syn)
            ]
            if syn_sum.empty:
                continue

            # preferred image = highest selectivity score
            pref_row = syn_df.loc[syn_df['image_selectivity_score'].idxmax()]
            session_n = syn_sum['session_#'].iloc[0]
            session_type = syn_sum['session_type'].iloc[0]
            if 'DMD1' in syn:
                syn_depth = syn_sum['dmd1_depth'].iloc[0]
            else:
                syn_depth = syn_sum['dmd2_depth'].iloc[0]

            rows.append({
                'session_id': sess,
                'session_#': session_n,
                'session_type':session_type,
                'dmd': dmd,
                'synapse_id': syn,
                'depth': syn_depth,
                'preferred_selectivity': pref_row['image_selectivity_score'],
                'preferred_early_slope': pref_row['early_slope'],
                'preferred_overall_slope':pref_row['overall_slope'],
                'median_abs_early_slope': np.median(np.abs(syn_df['early_slope'])),
                'mean_abs_early_slope': np.mean(np.abs(syn_df['early_slope'])),
            })

plot_df = pd.DataFrame(rows)

plot_df = plot_df[plot_df['preferred_selectivity'] > 0].copy()
plot_df['log10_preferred_selectivity'] = np.log10(abs(plot_df['preferred_selectivity']))
plot_df['asinh_preferred_early_slope'] = np.arcsinh(plot_df['preferred_early_slope'])
plot_df['asinh_overall_slope'] = np.arcsinh(plot_df['preferred_overall_slope'])

In [ ]:
plot_df

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))
sns.despine(ax=ax)

sns.scatterplot(
    data=plot_df,
    x='log10_preferred_selectivity',
    y='preferred_early_slope',
    hue='depth',
    hue_order=[25, 100, 200, 250],
    palette=palette,
    s=28,
    edgecolor='none',
    alpha=0.85,
    ax=ax,
)

ax.axhline(0, color='k', linestyle='--', linewidth=1)
ax.set_xlabel('log(selectivity score)')
ax.set_ylabel('Response slope')
ax.legend(title='Depth (µm)', frameon=False)

for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(2)

fig.tight_layout()

In [ ]:
plot_df

In [ ]:
depth_order = sorted(plot_df['depth'].dropna().unique())

fig,ax=plt.subplots(figsize=(4,4))

sns.despine()

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

sns.stripplot(data=plot_df,x='depth',y='preferred_overall_slope',hue='depth',palette=cp,legend=False,size=5)

medians = (
    plot_df.groupby('depth', observed=False)['preferred_overall_slope']
    .median()
    .reindex(depth_order)
    .values
)
ax.plot(range(len(depth_order)), medians, color='k', lw=3, marker='o', zorder=20)

ax.set_xlabel('Depth (\u03BCm) from pia')
ax.set_ylabel('Response Slope')
ax.set_ylim(-80,80)
for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)
ax.set_title('Average Response Slope')
fig.tight_layout()

filen = 'Avg_resp_slope_depth'
save_figure(fig,os.path.join(savepath,filen),formats=['.pdf','.png'],dpi=300)

In [ ]:
# ------------------------------------------------------------
# Pairwise depth comparisons
# ------------------------------------------------------------
depth_order = [25, 100, 200, 250]

pairwise_rows = []

for d1, d2 in itertools.combinations(depth_order, 2):
    x = plot_df.loc[plot_df["depth"] == d1, "preferred_overall_slope"].dropna().to_numpy()
    y = plot_df.loc[plot_df["depth"] == d2, "preferred_overall_slope"].dropna().to_numpy()

    if len(x) == 0 or len(y) == 0:
        continue

    stat, p = mannwhitneyu(x, y, alternative="two-sided")

    pairwise_rows.append(
        {
            "depth_1": d1,
            "depth_2": d2,
            "n_1": len(x),
            "n_2": len(y),
            "median_1": np.median(x),
            "median_2": np.median(y),
            "mean_1": np.mean(x),
            "mean_2": np.mean(y),
            "mw_u": stat,
            "p_uncorrected": p,
        }
    )

pairwise_stats = pd.DataFrame(pairwise_rows)

# ------------------------------------------------------------
# Multiple-comparisons correction
# ------------------------------------------------------------
reject, p_fdr, _, _ = multipletests(
    pairwise_stats["p_uncorrected"].values,
    alpha=0.05,
    method="fdr_bh",
)

pairwise_stats["p_fdr_bh"] = p_fdr
pairwise_stats["significant_fdr_bh"] = reject

pairwise_stats = pairwise_stats.sort_values("p_fdr_bh").reset_index(drop=True)

pairwise_stats

In [ ]:
session_df

In [ ]:
session_df = plot_df[(plot_df['session_#']>1)
                     &(plot_df['preferred_overall_slope']<0)
                    ]
fig,ax = plt.subplots(figsize=(6,4))
sns.despine()
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
for i,depth in enumerate(session_df['depth'].unique()):
    dft = session_df[session_df['depth']==depth]
    mean = []
    sem = []
    for sess in sorted(dft['session_#'].unique()):
        sess_type = dft['session_type'][dft['session_#']==sess].unique()
        
        sess_mean = np.mean(dft['preferred_overall_slope'][dft['session_#']==sess])
        sess_sem = np.std(dft['preferred_overall_slope'][dft['session_#']==sess])/np.sqrt(len(dft[dft['session_#']==sess]))
        
        mean.append(sess_mean)
        sem.append(sess_sem)
    ax.plot(mean,color=cp[i],lw=2,marker='o',label=f"{depth} \u03BCm")
    ax.fill_between(range(len(sem)),np.array(mean)+np.array(sem),np.array(mean)-np.array(sem),color=cp[i],alpha=0.2,zorder=0)
ax.set_xlabel('Session type')
ax.set_ylabel('Image response slope')

for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)

ax.set_xticks(range(6))
ax.set_xticklabels(['F','F','F','N','N+','N+'],fontsize=15)
    
ax.legend(frameon=False,fontsize=12)
ax.set_title('Session-wise avg. response slope')

ax.set_ylim(-20,20)

fig.tight_layout()

filen = 'Session_slope'
save_figure(fig,os.path.join(savepath,filen),formats = ['.pdf','.png'],dpi= 300)

In [ ]:
session_df = plot_df[(plot_df['session_#']>1)
                     &(plot_df['preferred_overall_slope']>0)
                    ]
fig,ax = plt.subplots(figsize=(6,4))
sns.despine()
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
for i,depth in enumerate(session_df['depth'].unique()):
    dft = session_df[session_df['depth']==depth]
    mean = []
    sem = []
    for sess in sorted(dft['session_#'].unique()):
        sess_type = dft['session_type'][dft['session_#']==sess].unique()
        
        sess_mean = np.mean(dft['preferred_overall_slope'][dft['session_#']==sess])
        sess_sem = np.std(dft['preferred_overall_slope'][dft['session_#']==sess])/np.sqrt(len(dft[dft['session_#']==sess]))
        
        mean.append(sess_mean)
        sem.append(sess_sem)
    ax.plot(mean,color=cp[i],lw=2,marker='o',label=f"{depth} \u03BCm")
    ax.fill_between(range(len(sem)),np.array(mean)+np.array(sem),np.array(mean)-np.array(sem),color=cp[i],alpha=0.2,zorder=0)
ax.set_xlabel('Session type')
ax.set_ylabel('Image response slope')

for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)

ax.set_xticks(range(6))
ax.set_xticklabels(['F','F','F','N','N+','N+'],fontsize=15)
    
ax.legend(frameon=False,fontsize=12)
ax.set_title('Session-wise avg. response slope')

ax.set_ylim(-20,20)

fig.tight_layout()

filen = 'Session_slope'
save_figure(fig,os.path.join(savepath,filen),formats = ['.pdf','.png'],dpi= 300)

In [ ]:
responses25 = []
responses100 = []
responses200 = []
responses250 = []

depth_map = {25:cp[0],100:cp[1],200:cp[2],250:cp[3]}

for sess in seq_per_image['session_id'].unique():
    sess_df = seq_per_image[seq_per_image['session_id']==sess]
    
    for dmd in sess_df['dmd'].unique():
        dmd_df = sess_df[sess_df['dmd']==dmd]
        
        for syn in dmd_df['synapse_id'].unique():
            syn_df = dmd_df[dmd_df['synapse_id']==syn].sort_values('overall_slope')
            
            syn_sum = seq_summary[(seq_summary['session_id']==sess)&(seq_summary['synapse_id']==syn)]
            
            if 'DMD1' in syn:
                syn_depth = syn_sum['dmd1_depth'].values[0]
            else:
                syn_depth = syn_sum['dmd2_depth'].values[0]
                
#             depth_cols.append(depth_map[syn_depth])
            if syn_depth == 25:
                responses25.append(syn_df['overall_slope'].values)
            elif syn_depth == 100:
                responses100.append(syn_df['overall_slope'].values)
            elif syn_depth == 200:
                responses200.append(syn_df['overall_slope'].values)
            elif syn_depth == 250:
                responses250.append(syn_df['overall_slope'].values)

In [ ]:
resps = np.array(responses250)

fig,ax=plt.subplots(figsize=(4,8))
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

first_vals = resps[:, 0]
last_vals = resps[:, -1]

row_order = np.lexsort((first_vals, -np.abs(last_vals)))
resps_sorted = resps[row_order]
# sns.heatmap(np.sort(np.sort(resps,axis=0)[::-1]),vmin=-20,vmax=20,cmap='viridis',cbar_kws={'shrink':0.5})
sns.heatmap(resps_sorted,vmin=-20,vmax=20,cmap='viridis',cbar_kws={'shrink':0.5})

ax.set_ylabel('Synapses')
ax.set_xlabel('Image (slope sorted)')
ax.set_title(f'250 \u03BCm (n = {len(resps)})')
fig.tight_layout()

filen = 'slope_heatmap_250'
# save_figure(fig,os.path.join(savepath,filen),formats=['.pdf','.png'],dpi=300)

In [ ]:
example = np.array(seq_per_image[seq_per_image['synapse_id']==syn].sort_values('overall_slope')['overall_slope'])

In [ ]:
responses = []

depth_map = {25:cp[0],100:cp[1],200:cp[2],250:cp[3]}

for sess in seq_per_image['session_id'].unique():
    sess_df = seq_per_image[seq_per_image['session_id']==sess]
    
    for dmd in sess_df['dmd'].unique():
        dmd_df = sess_df[sess_df['dmd']==dmd]
        
        for syn in dmd_df['synapse_id'].unique():
            syn_df = dmd_df[dmd_df['synapse_id']==syn].sort_values('overall_slope')
            responses.append(syn_df.iloc[:][['r0','rlast','rterminal']].values)
            

In [ ]:
np.shape(responses)

In [ ]:
fig,ax = plt.subplots()

for synapse in responses:
    if synapse[0][0]<0 and synapse[0][-1]>20:
        ax.plot(synapse)
ax.set_ylabel('Response slope')
ax.set_xlabel('Image ID')
ax.axhline(0,dashes=[3,3],color='k')
fig.tight_layout()

In [ ]:
fig,ax=plt.subplots(figsize=(4,4))

sns.despine()

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)
    
for r in responses[9]:
    ax.plot(r,marker='.')
    
ax.set_xticks([0,1,2])
ax.set_xticklabels(['First','Last','Change'])

ax.set_ylabel('\u0394F', rotation=0, labelpad=10)
ax.set_xlabel('Image in sequence')

# ax.set_title()

fig.tight_layout()